# SNAr RAG grounding corpus, and a no-RAG vs. RAG-grounded pilot

Builds and leakage-checks the retrieval-augmented-generation (RAG) safety
corpus used by the grounded agents in the SNAr safety case study: scrapes
LibreTexts organic-chemistry chapters and the Wikipedia article on
nucleophilic aromatic substitution with `trafilatura`, deduplicates by
pairwise text similarity, chunks and embeds the result with
`all-MiniLM-L6-v2`, and explicitly checks that the corpus contains none of
the mechanistic simulator's actual Arrhenius parameters or thresholds (so
retrieval must draw on general chemistry knowledge, not memorised numbers).
Writes `snar_grounding_corpus.json`.

Also pilots a small (5-seed, 12-iteration) GP-BO + Proposer/Critic/Verifier
comparison -- `bo_puro`, `multiagent_norag`, `multiagent_grounded` -- on the
same SNAr kinetic ODE simulator (Hone et al. 2017) used later by
`snar_safety_multiagent.ipynb`, to check whether grounding the Critic in
this corpus changes anything before scaling up.

**Scope note:** unlike the Buchwald-Hartwig and Direct Arylation notebooks,
this notebook's simulator and agent-loop code is **not** factored into the
shared `bayesllm` package. `snar_safety_multiagent.ipynb` reuses the same
kinetic model conceptually but reimplements it with real, verified
differences (its own objective wrapper, candidate encoding, and BO
proposer), which a programmatic diff (performed before any refactoring
decision was made) confirmed are not byte-identical between the two
notebooks. Forcing them into one shared implementation would risk silently
changing one of them; both are kept as their own, direct extraction of the
original notebook's logic instead. See `experiments/README.md` for detail.

Renamed from `dia14_grounded_sNAR.ipynb`.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

print("numpy:", np.__version__)

In [ ]:
# ============================================================
# dia14_grounding_snar.ipynb
# Cell 2 (verified against the real summit.benchmarks.snar source):
# SNAr kinetic simulator -- the "virtual lab"
# ============================================================

R_GAS_CONSTANT_SNAR = 8.314e-3            # kJ/(K·mol)
T_REF_OFFSET_SNAR = 273.71                # offset no estándar, fiel al original
T_REF_KELVIN_SNAR = 90.0 + T_REF_OFFSET_SNAR
REACTOR_VOLUME_ML_SNAR = 5.0              # mL
ETHANOL_DENSITY_SNAR = 0.789              # g/mL

# [substrate ("DNFB", MW=159.09 per the original code), pyrrolidine,
#  mono-A, mono-B (regioisomers), bis-product]
MOLAR_MASSES_SNAR = np.array([159.09, 71.12, 210.21, 210.21, 261.33])

ARRHENIUS_PARAMS_SNAR = {
    "k_a": (57.9, 33.3),
    "k_b": (2.70, 35.3),
    "k_c": (0.865, 38.9),
    "k_d": (1.63, 44.8),
}
ARRHENIUS_PREFACTOR_SNAR = 0.6


def compute_rate_constants_snar(temperature_celsius):
    T_kelvin = temperature_celsius + T_REF_OFFSET_SNAR
    return {
        name: ARRHENIUS_PREFACTOR_SNAR * k_ref * np.exp(
            -(E_a / R_GAS_CONSTANT_SNAR) * (1.0 / T_kelvin - 1.0 / T_REF_KELVIN_SNAR)
        )
        for name, (k_ref, E_a) in ARRHENIUS_PARAMS_SNAR.items()
    }


def snar_ode_rhs(t, C, k_a, k_b, k_c, k_d, C_initial):
    """Incluye la salvaguarda del original: reactivos casi agotados
    (< 1e-6 de su concentración inicial) se fuerzan a 0."""
    C = np.array(C, dtype=float)
    for i in (0, 1):
        if C[i] < 1e-6 * C_initial[i]:
            C[i] = 0.0
    C0, C1, C2, C3, C4 = C
    return [
        -(k_a + k_b) * C0 * C1,
        -(k_a + k_b) * C0 * C1 - k_c * C1 * C2 - k_d * C1 * C3,
        k_a * C0 * C1 - k_c * C1 * C2,
        k_b * C0 * C1 - k_d * C1 * C3,
        k_c * C1 * C2 + k_d * C1 * C3,
    ]


def compute_sty_and_efactor_snar(C_final, q_tot):
    """Mismos límites numéricos que el benchmark original."""
    sty = 6e4 / 1000 * MOLAR_MASSES_SNAR[2] * C_final[2] * q_tot / REACTOR_VOLUME_ML_SNAR
    sty = max(sty, 1e-6)

    if np.isclose(C_final[2], 0.0):
        e_factor = 1e3
    else:
        mass_unreacted = 1e-3 * sum(
            MOLAR_MASSES_SNAR[i] * C_final[i] * q_tot for i in range(5) if i != 2
        )
        e_factor = (q_tot * ETHANOL_DENSITY_SNAR + mass_unreacted) / (
            1e-3 * MOLAR_MASSES_SNAR[2] * C_final[2] * q_tot
        )
    e_factor = min(e_factor, 1e3)
    return sty, e_factor


def run_virtual_experiment_snar(temperature_celsius, tau_min, equiv_pldn,conc_dfnb_M,noise_level_pct, rng):
    """
    "Virtual lab": integra la cinética y devuelve tanto el resultado
    VERDADERO (sin ruido — solo para tu análisis, nunca para el LLM) como
    el OBSERVADO (con ruido de laboratorio aplicado a las concentraciones
    finales, no al STY directamente — así se propaga de forma no lineal
    a través de las fórmulas, igual que en el benchmark real).

    noise_level_pct: % de ruido relativo por especie (el parámetro
    `noise_level` real de Summit; por defecto ahí es 0).
    rng: numpy.random.Generator ya seedeado, para reproducibilidad por
    (seed, iteración) — mismo principio que torch.manual_seed en el resto
    del proyecto.
    """
    k = compute_rate_constants_snar(temperature_celsius)
    C_initial = np.array([conc_dfnb_M, equiv_pldn * conc_dfnb_M, 0.0, 0.0, 0.0])

    solution = solve_ivp(
        snar_ode_rhs, t_span=(0, tau_min), y0=C_initial,
        args=(k["k_a"], k["k_b"], k["k_c"], k["k_d"], C_initial),
        method="RK45", rtol=1e-8, atol=1e-10,
    )
    C_final_true = solution.y[:, -1]
    q_tot = REACTOR_VOLUME_ML_SNAR / tau_min

    true_sty, true_e_factor = compute_sty_and_efactor_snar(C_final_true, q_tot)

    C_final_noisy = C_final_true + (
        C_final_true * rng.normal(scale=noise_level_pct, size=5) / 100
    )
    C_final_noisy[C_final_noisy < 0] = 0.0
    observed_sty, observed_e_factor = compute_sty_and_efactor_snar(C_final_noisy, q_tot)

    return {
        "observed_sty": observed_sty, "observed_e_factor": observed_e_factor,
        "true_sty": true_sty, "true_e_factor": true_e_factor,
    }

In [ ]:
rng_check_snar = np.random.default_rng(0)
temperatures_to_test = np.linspace(30, 120, 20)
sty_values = [
    run_virtual_experiment_snar(T, tau_min=1.0, equiv_pldn=2.5, conc_dfnb_M=0.3,
                                 noise_level_pct=0.0, rng=rng_check_snar)["true_sty"]
    for T in temperatures_to_test
]

plt.figure(figsize=(6, 4))
plt.plot(temperatures_to_test, sty_values, marker="o")
plt.xlabel("Temperatura (°C)")
plt.ylabel("STY verdadero (kg/m³/h)")
plt.title("Simulador SNAr verificado: STY vs temperatura")
plt.grid(alpha=0.3)
plt.show()
print(f"Rango STY: {min(sty_values):.1f} - {max(sty_values):.1f} kg/m3/h "
      f"(el dominio original acota STY en [0, 13000])")

In [ ]:
NOISE_LEVEL_PCT_SNAR = 5.0  # % de ruido relativo por especie — punto de partida razonable
                             # and adjustable; the benchmark itself leaves it at 0 by default.

rng_demo_snar = np.random.default_rng(42)
true_reference_snar = run_virtual_experiment_snar(
    90, 1.0, 2.5, 0.3, noise_level_pct=0.0, rng=rng_demo_snar)["true_sty"]

noisy_observations_snar = [
    run_virtual_experiment_snar(90, 1.0, 2.5, 0.3,
                                 noise_level_pct=NOISE_LEVEL_PCT_SNAR, rng=rng_demo_snar)["observed_sty"]
    for _ in range(30)
]
print(f"STY verdadero (referencia): {true_reference_snar:.2f}")
print(f"STY observado (30 repeticiones, {NOISE_LEVEL_PCT_SNAR}% ruido): "
      f"media={np.mean(noisy_observations_snar):.2f}, std={np.std(noisy_observations_snar):.2f}")

In [ ]:
# ============================================================
# dia14_grounding_snar.ipynb
# Cell 5: imports for the grounding corpus (manual + literature)
# ============================================================

import requests
import trafilatura

print("Imports ready.")

In [ ]:
# ============================================================
# Cell 6: fetch and extract clean text from one LibreTexts SNAr chapter
# ============================================================

def fetch_clean_text_from_url(url):
    """
    Downloads a webpage and returns just its main article text, discarding
    navigation menus, sidebars, footers, etc.

    Two-step process:
    1. trafilatura.fetch_url(url) — downloads the raw HTML (similar to
       requests.get(url).text, but also handles some encoding edge cases
       automatically).
    2. trafilatura.extract(downloaded_html) — takes that raw HTML and
       returns just the clean article text as a plain string.

    They are two separate calls so you could also pass in HTML you already
    have from elsewhere (e.g. a cached copy) without re-downloading.

    Returns None if the download failed, or if trafilatura could not
    confidently identify a main content block on the page.
    """
    downloaded_html = trafilatura.fetch_url(url)
    if downloaded_html is None:
        return None
    return trafilatura.extract(downloaded_html)


snar_mechanism_url = (
    "https://chem.libretexts.org/Courses/Winona_State_University/"
    "Klein_and_Straumanis_Guided/19:_Aromatic_Substitution_Reactions/"
    "19.09:_Nucleophilic_Aromatic_Substitution"
)

snar_mechanism_text = fetch_clean_text_from_url(snar_mechanism_url)

if snar_mechanism_text:
    print(f"Extracted {len(snar_mechanism_text)} characters.\n")
    print(snar_mechanism_text[:1000])  # preview the first 1000 characters
else:
    print("Extraction failed — check the URL or try a different page.")

In [ ]:
# ============================================================
# Cell 7: strip LibreTexts (MindTouch) boilerplate from extracted text
# ============================================================
import re

def strip_libretexts_boilerplate(text):
    """
    LibreTexts pages inject two kinds of boilerplate as literal text
    inside the article body -- indistinguishable from real content by
    trafilatura's extraction heuristics, since they live in the same
    HTML container as the actual chapter text:

      1. A block of MathJax macro definitions (lines containing
         "\\newcommand"), used for equation rendering elsewhere on the
         page. This appears on EVERY LibreTexts page, not just this one,
         so this cleanup function will be reused for every URL we scrape.
      2. A small page-metadata header ("- Page ID" followed by a bare
         numeric ID).

    Approach: filter text line by line, dropping any line that matches
    either pattern. Since "\\newcommand" is a very distinctive string
    that would essentially never appear in real chemistry prose, a simple
    per-line filter is robust -- we don't need to know exactly how many
    macro-definition lines there are or where the block ends.
    """
    cleaned_lines = []
    for line in text.split("\n"):
        stripped_line = line.strip()
        if "newcommand" in stripped_line:
            continue
        if stripped_line == "- Page ID" or stripped_line.isdigit():
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines).strip()


snar_mechanism_text_clean = strip_libretexts_boilerplate(snar_mechanism_text)

print(f"Before cleanup: {len(snar_mechanism_text)} characters")
print(f"After cleanup:  {len(snar_mechanism_text_clean)} characters\n")
print(snar_mechanism_text_clean[:1500])

In [ ]:
# ============================================================
# Cell 8: fetch text from several LibreTexts SNAr chapters
# ============================================================
# We already validated the fetch+clean pipeline on one page (Cells 6-7).
# Now we apply it to a handful of other LibreTexts pages on the same
# topic, from different textbook adaptations -- several will overlap
# heavily in content (LibreTexts republishes course adaptations of the
# same underlying textbooks, e.g. McMurry), that's expected. We're just
# gathering candidates here; we'll decide together which ones to actually
# keep after seeing what came back.

snar_libretexts_urls = [
    "https://chem.libretexts.org/Courses/Winona_State_University/Klein_and_Straumanis_Guided/19:_Aromatic_Substitution_Reactions/19.09:_Nucleophilic_Aromatic_Substitution",
    "https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Map:_Organic_Chemistry_(McMurry)/16:_Chemistry_of_Benzene_-_Electrophilic_Aromatic_Substitution/16.06:_Nucleophilic_Aromatic_Substitution",
    "https://chem.libretexts.org/Courses/SUNY_Potsdam/Book:_Organic_Chemistry_II_(Walker)/17:_Other_Reactions_of_Aromatics/17.01:_Nucleophilic_aromatic_substitution",
    "https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Map:_Organic_Chemistry_(Vollhardt_and_Schore)/22:_Chemistry_of_the_Benzene_Substituents:_Alkylbenzenes_Phenols_and_Benzenamines/22.04:_Preparation__of_Phenols:__Nucleophilic__Aromatic__Substitution",
]

snar_pages_raw = {}
for url in snar_libretexts_urls:
    text = fetch_clean_text_from_url(url)
    snar_pages_raw[url] = text
    status = f"{len(text)} chars" if text else "FAILED"
    print(f"{status:>12}  {url.split('/')[-1]}")

In [ ]:
# ============================================================
# Cell 9: clean boilerplate from every fetched page, preview each
# ============================================================
snar_pages_clean = {}
for url, raw_text in snar_pages_raw.items():
    if raw_text is None:
        continue  # skip pages where the fetch itself failed
    clean_text = strip_libretexts_boilerplate(raw_text)
    snar_pages_clean[url] = clean_text
    label = url.split("/")[-1]
    print(f"\n=== {label} ({len(clean_text)} chars, was {len(raw_text)}) ===")
    print(clean_text[:400])

In [ ]:
# ============================================================
# Cell 10: check how much the 4 scraped pages actually overlap
# ============================================================
from difflib import SequenceMatcher

urls_list = list(snar_pages_clean.keys())
print("Pairwise text similarity (0 = completely different, 1 = identical):\n")
for i in range(len(urls_list)):
    for j in range(i + 1, len(urls_list)):
        text_a = snar_pages_clean[urls_list[i]]
        text_b = snar_pages_clean[urls_list[j]]
        similarity = SequenceMatcher(None, text_a, text_b).ratio()
        label_a = urls_list[i].split("/")[-1][:25]
        label_b = urls_list[j].split("/")[-1][:25]
        print(f"{similarity:.2f}   {label_a}  vs  {label_b}")

In [ ]:
# ============================================================
# Cell 11: keep only the non-redundant pages -> final "general mechanism" corpus
# ============================================================
urls_to_keep_snar_mechanism = [
    "https://chem.libretexts.org/Courses/SUNY_Potsdam/Book:_Organic_Chemistry_II_(Walker)/17:_Other_Reactions_of_Aromatics/17.01:_Nucleophilic_aromatic_substitution",
    "https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Map:_Organic_Chemistry_(Vollhardt_and_Schore)/22:_Chemistry_of_the_Benzene_Substituents:_Alkylbenzenes_Phenols_and_Benzenamines/22.04:_Preparation__of_Phenols:__Nucleophilic__Aromatic__Substitution",
]

snar_general_mechanism_corpus = {
    url: snar_pages_clean[url] for url in urls_to_keep_snar_mechanism
}

total_chars = sum(len(t) for t in snar_general_mechanism_corpus.values())
print(f"Final general-mechanism corpus: {len(snar_general_mechanism_corpus)} pages, "
      f"{total_chars} characters total")

In [ ]:
# ============================================================
# Cell 12: fetch the OCLUE page (different textbook) on SNAr,
# check it against the current corpus for redundancy
# ============================================================
oclue_url = (
    "https://chem.libretexts.org/Bookshelves/Organic_Chemistry/"
    "OCLUE:_Organic_Chemistry_Life_the_Universe_and_Everything_(Copper_and_Klymkowsky)/"
    "08:_Conjugated_compounds_and_aromaticity/"
    "8.12:_Nucleophilic_Substitutions_on_Aromatic_Systems-"
    "_Expanding_the_range_of_potential_substitution_products"
)

oclue_raw = fetch_clean_text_from_url(oclue_url)
oclue_clean = strip_libretexts_boilerplate(oclue_raw) if oclue_raw else None

if oclue_clean:
    print(f"Extracted {len(oclue_clean)} chars\n")
    for existing_url, existing_text in snar_general_mechanism_corpus.items():
        sim = SequenceMatcher(None, oclue_clean, existing_text).ratio()
        print(f"Similarity to {existing_url.split('/')[-1][:30]}: {sim:.2f}")
    print()
    print(oclue_clean[:600])
else:
    print("Fetch/extraction failed.")

In [ ]:
# ============================================================
# Cell 18: commit the OCLUE page into the grounding corpus,
# and persist the finished corpus to disk
# ============================================================
snar_general_mechanism_corpus[oclue_url] = oclue_clean

print(f"Corpus final: {len(snar_general_mechanism_corpus)} páginas")
for url, text in snar_general_mechanism_corpus.items():
    print(f"  - {url.split('/')[-1][:60]}  ({len(text)} chars)")

import json
corpus_path = "snar_grounding_corpus.json"
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(snar_general_mechanism_corpus, f, ensure_ascii=False, indent=2)

print(f"\nGuardado en {corpus_path}")

In [ ]:
# ============================================================
# Cell 14: leakage check -- verify the grounding corpus does NOT
# contain the exact parameters that generate the simulator
# ============================================================
leakage_indicators = {
    "Arrhenius k_a (57.9)": "57.9",
    "Arrhenius k_b (2.70)": "2.70",
    "Arrhenius k_c (0.865)": "0.865",
    "Arrhenius k_d (1.63)": "1.63",
    "Ea k_a (33.3)": "33.3",
    "Ea k_b (35.3)": "35.3",
    "Ea k_c (38.9)": "38.9",
    "Ea k_d (44.8)": "44.8",
    "T_ref offset (273.71)": "273.71",
    "DNFB molar mass (159.09)": "159.09",
    "pirrolidina molar mass (71.12)": "71.12",
    "prefactor Arrhenius (0.6)": "0.6",
    "'DNFB' literal": "DNFB",
    "'Hone' (autor del paper fuente)": "Hone",
    "'STY' (space-time yield)": "STY",
    "'E-factor'": "E-factor",
}

full_corpus_text = "\n".join(snar_general_mechanism_corpus.values())

print(f"Corpus total: {len(full_corpus_text)} characters\n")
print("Búsqueda de indicadores de fuga:\n")
hits = 0
for label, needle in leakage_indicators.items():
    found = needle.lower() in full_corpus_text.lower()
    marker = "⚠️  ENCONTRADO" if found else "OK -- ausente"
    print(f"  [{marker}] {label}")
    if found:
        hits += 1

print(f"\n{'⚠️  Revisar antes de seguir' if hits else '✅ Sin indicios de fuga -- el corpus es seguro para el Critic'}")

In [ ]:
# ============================================================
# Cell 15: split the grounding corpus into retrievable chunks
# ============================================================
def chunk_text(text, chunk_size=800, overlap=150):
    """
    Splits text into overlapping chunks of ~chunk_size characters.
    Overlap keeps a sentence/idea that straddles a chunk boundary from
    losing context in both halves.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


snar_corpus_chunks = []  # lista de dicts: {"source": url, "text": chunk}
for url, text in snar_general_mechanism_corpus.items():
    for chunk in chunk_text(text):
        snar_corpus_chunks.append({"source": url, "text": chunk})

print(f"{len(snar_general_mechanism_corpus)} páginas -> {len(snar_corpus_chunks)} chunks")
for c in snar_corpus_chunks[:3]:
    print(f"\n[{c['source'].split('/')[-1][:40]}]")
    print(c['text'][:200])

In [ ]:
# ============================================================
# Cell 16: embed every chunk once, cache the vectors
# ============================================================
from sentence_transformers import SentenceTransformer  # pip install sentence-transformers, si hace falta

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, corre en CPU sin problema

chunk_texts = [c["text"] for c in snar_corpus_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True)

print(f"Embeddings: {chunk_embeddings.shape}")  # (n_chunks, 384)

In [ ]:
# ============================================================
# Cell 17: retrieval function -- given a query, return the top-k
# most relevant chunks from the grounding corpus
# ============================================================
import numpy as np

def retrieve_grounding_context(query, top_k=3):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    similarities = chunk_embeddings @ query_embedding  # producto escalar = similitud coseno (vectores normalizados)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**snar_corpus_chunks[i], "similarity": float(similarities[i])}
        for i in top_indices
    ]



In [ ]:
# ============================================================
# Cell 18: diagnose attribution/citation boilerplate leaking
# into the corpus chunks (found via the retrieval smoke test)
# ============================================================
import re

citation_pattern = re.compile(r"(License:|CC BY|LibreTexts\.|Project: Chemistry LibreTexts)", re.IGNORECASE)

for url, text in snar_general_mechanism_corpus.items():
    lines = text.split("\n")
    matches = [l for l in lines if citation_pattern.search(l)]
    print(f"\n=== {url.split('/')[-1][:50]} ===")
    print(f"{len(matches)} líneas sospechosas de {len(lines)} totales")
    for m in matches:
        print(f"  > {m[:150]}")

In [ ]:
# ============================================================
# Cell 19: strip the attribution/citation line found in Cell 18
# from the corpus, and re-save
# ============================================================
citation_pattern = re.compile(r"(License:|CC BY|Project: Chemistry LibreTexts|Authored by:)", re.IGNORECASE)

def strip_citation_lines(text):
    return "\n".join(
        line for line in text.split("\n") if not citation_pattern.search(line)
    ).strip()

for url in snar_general_mechanism_corpus:
    before = len(snar_general_mechanism_corpus[url])
    snar_general_mechanism_corpus[url] = strip_citation_lines(snar_general_mechanism_corpus[url])
    after = len(snar_general_mechanism_corpus[url])
    if before != after:
        print(f"{url.split('/')[-1][:50]}: {before} -> {after} chars ({before - after} recortados)")

with open("snar_grounding_corpus.json", "w", encoding="utf-8") as f:
    json.dump(snar_general_mechanism_corpus, f, ensure_ascii=False, indent=2)
print("\nCorpus actualizado guardado.")

In [ ]:
# ============================================================
# Cell 26: fetch the Wikipedia SNAr article -- avoids the MindTouch
# FIB-widget extraction failure seen on the Nassau CC page, and uses
# 2,4-dinitrochlorobenzene as its worked example (same substitution
# pattern as our DNFB substrate)
# ============================================================
wikipedia_snar_url = "https://en.wikipedia.org/wiki/Nucleophilic_aromatic_substitution"

wikipedia_snar_raw = fetch_clean_text_from_url(wikipedia_snar_url)
wikipedia_snar_clean = (
    strip_citation_lines(strip_libretexts_boilerplate(wikipedia_snar_raw))
    if wikipedia_snar_raw else None
)

if wikipedia_snar_clean:
    print(f"Extracted {len(wikipedia_snar_clean)} chars\n")
    for existing_url, existing_text in snar_general_mechanism_corpus.items():
        sim = SequenceMatcher(None, wikipedia_snar_clean, existing_text).ratio()
        print(f"Similarity to {existing_url.split('/')[-1][:30]}: {sim:.2f}")
    print()
    print(wikipedia_snar_clean[:600])
else:
    print("Fetch/extraction failed.")

In [ ]:
# ============================================================
# Cell 27: leakage check on the new Wikipedia page before
# committing it to the corpus -- same indicators as Cell 14
# ============================================================
print("Chequeo de fuga sobre el artículo de Wikipedia:\n")
hits = 0
for label, needle in leakage_indicators.items():
    found = needle.lower() in wikipedia_snar_clean.lower()
    marker = "⚠️  ENCONTRADO" if found else "OK -- ausente"
    print(f"  [{marker}] {label}")
    if found:
        hits += 1

print(f"\n{'⚠️  Revisar antes de añadir' if hits else '✅ Sin indicios de fuga -- seguro para el corpus'}")

In [ ]:
# ============================================================
# Cell 28: commit the Wikipedia page to the corpus and re-save
# ============================================================
snar_general_mechanism_corpus[wikipedia_snar_url] = wikipedia_snar_clean

print(f"Corpus final: {len(snar_general_mechanism_corpus)} páginas")
for url, text in snar_general_mechanism_corpus.items():
    print(f"  - {url.split('/')[-1][:60]}  ({len(text)} chars)")

with open("snar_grounding_corpus.json", "w", encoding="utf-8") as f:
    json.dump(snar_general_mechanism_corpus, f, ensure_ascii=False, indent=2)

print("\nGuardado en snar_grounding_corpus.json")

In [ ]:
# ============================================================
# Cell 29: re-chunk the corpus now that it has 4 pages (was 3
# when we first built the retrieval index in Cell 15)
# ============================================================
snar_corpus_chunks = []
for url, text in snar_general_mechanism_corpus.items():
    for chunk in chunk_text(text):
        snar_corpus_chunks.append({"source": url, "text": chunk})

print(f"{len(snar_general_mechanism_corpus)} páginas -> {len(snar_corpus_chunks)} chunks")

In [ ]:
# ============================================================
# Cell 30: re-embed all chunks of the finalized corpus
# ============================================================
chunk_texts = [c["text"] for c in snar_corpus_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True)

print(f"Embeddings: {chunk_embeddings.shape}")

In [ ]:
# ============================================================
# dia14_grounding_snar.ipynb
# Cell 32: BO + multiagent imports, and the 4D search space bounds
# -- verified literally against summit.benchmarks.snar's
# _setup_domain() (pasted and confirmed, not reconstructed)
# ============================================================
import ollama
import json
import torch
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.acquisition import LogExpectedImprovement
# order: [tau, equiv_pldn, conc_dfnb, temperature] -- same order as the
# literal _setup_domain() definition just verified, so that it's
# trivial to compare against the source if needed
BOUNDS_SNAR = torch.tensor([
    [0.5, 1.0, 0.1, 30.0],    # lower bounds: tau, equiv_pldn, conc_dfnb, temperature
    [2.0, 5.0, 0.5, 120.0],   # upper bounds
], dtype=torch.float64)

NOMBRES_DIMENSIONES_SNAR = ["tau", "equiv_pldn", "conc_dfnb", "temperature"]
N_DIMS_SNAR = BOUNDS_SNAR.shape[1]        # 4
N_INIT_SNAR = 2 * N_DIMS_SNAR             # 8, mismo criterio que dia13 (Direct Arylation)

print(f"Espacio de búsqueda SNAr: {N_DIMS_SNAR}D")
for nombre, lo, hi in zip(NOMBRES_DIMENSIONES_SNAR, BOUNDS_SNAR[0].tolist(), BOUNDS_SNAR[1].tolist()):
    print(f"  {nombre}: [{lo}, {hi}]")

In [ ]:
# ============================================================
# Cell 33: dict<->tensor helpers, and an objective wrapper that
# calls the simulator by keyword (Cell 2), not position -- avoids
# any mismatch between BOUNDS_SNAR's column order and the real
# function signature
# ============================================================
def tensor_a_dict_snar(x_tensor):
    """x_tensor: (4,) o (1,4), orden BOUNDS_SNAR [tau, equiv_pldn, conc_dfnb, temperature]"""
    x_flat = x_tensor.flatten()
    return {
        "tau": x_flat[0].item(),
        "equiv_pldn": x_flat[1].item(),
        "conc_dfnb": x_flat[2].item(),
        "temperature": x_flat[3].item(),
    }


def dict_a_tensor_snar(d):
    return torch.tensor([[
        d["tau"], d["equiv_pldn"], d["conc_dfnb"], d["temperature"]
    ]], dtype=torch.float64)


def objetivo_snar(d, noise_level_pct, rng):
    resultado = run_virtual_experiment_snar(
        temperature_celsius=d["temperature"],
        tau_min=d["tau"],
        equiv_pldn=d["equiv_pldn"],
        conc_dfnb_M=d["conc_dfnb"],
        noise_level_pct=noise_level_pct,
        rng=rng,
    )
    return resultado["observed_sty"]


# Quick test -- should give an STY similar to the ~2928 from Cell 4
rng_test = np.random.default_rng(0)
test_dict = tensor_a_dict_snar(torch.tensor([[1.0, 2.5, 0.3, 90.0]]))
print(test_dict)
print(f"STY observado: {objetivo_snar(test_dict, noise_level_pct=NOISE_LEVEL_PCT_SNAR, rng=rng_test):.2f}")

In [ ]:
# ============================================================
# Cell 34: Proposer/Critic/Verifier prompts adapted to the 4D
# SNAr space -- same architecture as dia5_multiagent.ipynb v2
# (deterministic bounds/redundancy + LLM coherence judgment),
# plus grounding injected into the Critic
# ============================================================
PROPOSER_PROMPT_SNAR = """You are an expert chemist acting as a creative experimentalist,
specialized in nucleophilic aromatic substitution (SNAr) reactions.

You receive a candidate experiment proposed by Bayesian Optimization and the full
experimental history for a SNAr reaction: 2,4-dinitrofluorobenzene (DNFB) reacting with
pyrrolidine in a continuous flow reactor, forming two mono-substituted regioisomers that
can further react to a bis-substituted product. Your job is to enrich the candidate with
chemical reasoning: explain why it could work, what chemical mechanisms support it (e.g.
how the electron-withdrawing nitro groups activate the ring towards nucleophilic attack,
how temperature/residence time/pyrrolidine equivalents/concentration affect conversion and
selectivity), and whether the direction makes sense given the history.

Respond ONLY with a valid JSON object:
{
    "tau": <number, residence time in minutes>,
    "equiv_pldn": <number, equivalents of pyrrolidine>,
    "conc_dfnb": <number, concentration of DNFB in M>,
    "temperature": <number, temperature in Celsius>,
    "reasoning": "<string explaining the chemical rationale>"
}
No additional text, no markdown, no units inside values."""


CRITIC_PROMPT_SNAR_V2 = """You are a rigorous chemistry expert reviewing an experimental
proposal for a nucleophilic aromatic substitution (SNAr) reaction (DNFB + pyrrolidine).
The proposal has ALREADY passed automated safety and feasibility checks (bounds and
redundancy are verified separately and are NOT your concern).

You will be given relevant excerpts from chemistry reference material about the SNAr
mechanism, retrieved specifically for this proposal. Ground your judgment in them -- do
not rely solely on your own memorized knowledge.

Your ONLY job is to evaluate whether the chemical reasoning provided is coherent,
plausible, and consistent with both the experimental history and the reference material.
Reject only if the reasoning is contradictory, nonsensical, or unsupported by the
data/reference shown.

Respond ONLY with a valid JSON object:
{
    "approved": <true or false>,
    "rejection_reason": "<string if rejected, else null>",
    "concern_level": "<low|medium|high>"
}
No additional text, no markdown."""


VERIFIER_PROMPT_SNAR = """You are the lead scientist overseeing a sequential experiment
campaign on a SNAr reaction (DNFB + pyrrolidine). You receive a proposal that has passed
initial safety and coherence review. Your job is to make the final decision: confirm the
experiment as-is, make a minor adjustment if needed, and log your reasoning for
interpretability.

Respond ONLY with a valid JSON object:
{
    "tau": <number>,
    "equiv_pldn": <number>,
    "conc_dfnb": <number>,
    "temperature": <number>,
    "adjustment_made": <true or false>,
    "final_reasoning": "<string: why this experiment is worth running>"
}
No additional text, no markdown, no units inside values."""

In [ ]:
# ============================================================
# Cell 35: deterministic bounds/redundancy checks, generalized
# from dia5_multiagent.ipynb to the 4 SNAr dimensions
# ============================================================
def verificar_bounds_snar(propuesta, bounds):
    valores = [propuesta[n] for n in NOMBRES_DIMENSIONES_SNAR]
    for valor, nombre, lo, hi in zip(
        valores, NOMBRES_DIMENSIONES_SNAR, bounds[0].tolist(), bounds[1].tolist()
    ):
        if valor < lo or valor > hi:
            return False, f"{nombre}={valor} fuera de [{lo}, {hi}]"
    return True, None


def normalizar_snar(propuesta, bounds):
    valores = torch.tensor([propuesta[n] for n in NOMBRES_DIMENSIONES_SNAR])
    return (valores - bounds[0]) / (bounds[1] - bounds[0])


def verificar_redundancia_snar(propuesta, historial, bounds, umbral=0):
    x_nuevo = normalizar_snar(propuesta, bounds)
    for exp in historial:
        x_previo = normalizar_snar(exp, bounds)
        distancia = torch.norm(x_nuevo - x_previo).item()
        if distancia < umbral:
            return False, f"Demasiado similar a experimento previo (dist={distancia:.3f})"
    return True, None


# Quick test
test_ok = {"tau": 1.0, "equiv_pldn": 2.5, "conc_dfnb": 0.3, "temperature": 90}
test_fuera = {"tau": 1.0, "equiv_pldn": 2.5, "conc_dfnb": 0.3, "temperature": 150}
print(verificar_bounds_snar(test_ok, BOUNDS_SNAR))
print(verificar_bounds_snar(test_fuera, BOUNDS_SNAR))

In [ ]:
def parsear_respuesta(raw):
    if '</think>' in raw:
        raw = raw.split('</think>', 1)[1]
    clean = raw.strip()
    if clean.startswith("'''"):
        clean = clean.split("'''")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    return json.loads(clean.strip(), strict=False)


def call_ollama_json_snar(system_prompt, user_prompt, seed, model='qwen3:30b', max_intentos=3):
    system_prompt = system_prompt + "\n\n/no_think"
    ultimo_raw = None
    for intento in range(max_intentos):
        respuesta = ollama.chat(
            model=model,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt}
            ],
            think=False,
            options={'temperature': OLLAMA_TEMPERATURE, 'seed': seed + intento}
        )
        ultimo_raw = respuesta['message']['content']
        try:
            return parsear_respuesta(ultimo_raw)
        except json.JSONDecodeError:
            continue
    raise ValueError(f"LLM no devolvió JSON válido tras {max_intentos} intentos. Última respuesta:\n{ultimo_raw}")


def formatear_historial_snar(historial):
    texto = ""
    for i, exp in enumerate(historial):
        texto += (f"Experimento{i+1}: tau={exp['tau']:.2f}min, "
                  f"equiv_pldn={exp['equiv_pldn']:.2f}, conc_dfnb={exp['conc_dfnb']:.2f}M, "
                  f"temperature={exp['temperature']:.1f}C, sty={exp['sty']:.1f}\n")
    return texto


CRITIC_PROMPT_SNAR_NO_RAG = """You are a rigorous chemistry expert reviewing an experimental
proposal for a nucleophilic aromatic substitution (SNAr) reaction (DNFB + pyrrolidine).
The proposal has ALREADY passed automated safety and feasibility checks (bounds and
redundancy are verified separately and are NOT your concern).

Your ONLY job is to evaluate whether the chemical reasoning provided is coherent,
plausible, and consistent with the experimental history. Reject only if the reasoning is
contradictory, nonsensical, or unsupported by the data shown.

Respond ONLY with a valid JSON object:
{
    "approved": <true or false>,
    "rejection_reason": "<string if rejected, else null>",
    "concern_level": "<low|medium|high>"
}
No additional text, no markdown."""


OLLAMA_TEMPERATURE = 0.0

def _ollama_seed(seed_experimento, iteracion, rol_id):
    return seed_experimento * 1000 + iteracion * 10 + rol_id


def llamar_proposer_snar(candidato_bo, historial, seed_experimento=0, iteracion=0):
    contexto = formatear_historial_snar(historial)
    contexto += (f"\nBayesian Optimisation suggests testing: "
                 f"tau={candidato_bo['tau']:.2f}, equiv_pldn={candidato_bo['equiv_pldn']:.2f}, "
                 f"conc_dfnb={candidato_bo['conc_dfnb']:.2f}, temperature={candidato_bo['temperature']:.1f}. "
                 f"Enrich this proposal with your chemical reasoning.")
    seed = _ollama_seed(seed_experimento, iteracion, 0)
    return call_ollama_json_snar(PROPOSER_PROMPT_SNAR, contexto, seed)


def llamar_critic_snar_v2(propuesta, historial, bounds, seed_experimento=0, iteracion=0):
    bounds_ok, razon_bounds = verificar_bounds_snar(propuesta, bounds)
    if not bounds_ok:
        return {'approved': False, 'rejection_reason': razon_bounds,
                'concern_level': 'high', 'capa_rechazo': 'deterministica',
                'grounding_usado': []}

    redundancia_ok, razon_redundancia = verificar_redundancia_snar(propuesta, historial, bounds)
    if not redundancia_ok:
        return {'approved': False, 'rejection_reason': razon_redundancia,
                'concern_level': 'medium', 'capa_rechazo': 'deterministica',
                'grounding_usado': []}

    query_grounding = (
        f"SNAr reaction at temperature={propuesta['temperature']}C, "
        f"residence time={propuesta['tau']} min, {propuesta['equiv_pldn']} equivalents "
        f"of pyrrolidine, {propuesta['conc_dfnb']} M DNFB. Reasoning: {propuesta['reasoning']}"
    )
    fragmentos = retrieve_grounding_context(query_grounding, top_k=3)
    bloque_grounding = "\n\n".join(
        f"[Reference excerpt, similarity={f['similarity']:.2f}]\n{f['text']}" for f in fragmentos
    )

    contexto = formatear_historial_snar(historial)
    contexto += (f"\nProposed experiment: tau={propuesta['tau']}, equiv_pldn={propuesta['equiv_pldn']}, "
                f"conc_dfnb={propuesta['conc_dfnb']}, temperature={propuesta['temperature']}. "
                f"Reasoning given: {propuesta['reasoning']}\n\n"
                f"Relevant reference material:\n{bloque_grounding}\n\n"
                f"Evaluate the coherence of this reasoning against the reference material and history.")

    seed = _ollama_seed(seed_experimento, iteracion, 1)
    veredicto = call_ollama_json_snar(CRITIC_PROMPT_SNAR_V2, contexto, seed)
    veredicto['capa_rechazo'] = 'llm' if not veredicto['approved'] else None
    veredicto['grounding_usado'] = [f['source'] for f in fragmentos]
    return veredicto


def llamar_critic_snar_no_rag(propuesta, historial, bounds, seed_experimento=0, iteracion=0):
    bounds_ok, razon_bounds = verificar_bounds_snar(propuesta, bounds)
    if not bounds_ok:
        return {'approved': False, 'rejection_reason': razon_bounds,
                'concern_level': 'high', 'capa_rechazo': 'deterministica'}

    contexto = formatear_historial_snar(historial)
    contexto += (f"\nProposed experiment: tau={propuesta['tau']}, equiv_pldn={propuesta['equiv_pldn']}, "
                f"conc_dfnb={propuesta['conc_dfnb']}, temperature={propuesta['temperature']}. "
                f"Reasoning given: {propuesta['reasoning']}\n"
                f"Evaluate the coherence of this reasoning.")

    seed = _ollama_seed(seed_experimento, iteracion, 1)
    veredicto = call_ollama_json_snar(CRITIC_PROMPT_SNAR_NO_RAG, contexto, seed)
    veredicto['capa_rechazo'] = 'llm' if not veredicto['approved'] else None
    return veredicto


def llamar_verifier_snar(propuesta, historial, seed_experimento=0, iteracion=0):
    contexto = formatear_historial_snar(historial)
    contexto += (f"\nApproved proposal: tau={propuesta['tau']}, equiv_pldn={propuesta['equiv_pldn']}, "
                f"conc_dfnb={propuesta['conc_dfnb']}, temperature={propuesta['temperature']}. "
                f"Make your final decision.")
    seed = _ollama_seed(seed_experimento, iteracion, 2)
    return call_ollama_json_snar(VERIFIER_PROMPT_SNAR, contexto, seed)


def clip_a_bounds_snar(propuesta_dict, bounds):
    recortado = dict(propuesta_dict)
    fuera_de_rango = []
    for nombre, lo, hi in zip(NOMBRES_DIMENSIONES_SNAR, bounds[0].tolist(), bounds[1].tolist()):
        valor = recortado[nombre]
        if valor < lo or valor > hi:
            fuera_de_rango.append(f"{nombre}={valor:.3f} -> {max(lo, min(hi, valor)):.3f}")
            recortado[nombre] = max(lo, min(hi, valor))
    return recortado, fuera_de_rango

In [ ]:
# ============================================================
# Cell 37: fit the GP over the unit cube [0,1]^4 (normalizing
# with BOUNDS_SNAR) instead of in physical units -- avoids the
# InputDataWarning and makes the 4 differently-scaled dimensions
# comparable. LogExpectedImprovement instead of EI, per the
# direct recommendation of botorch's warning (same API, numerically better).
# ============================================================
def proponer_candidato_bo_snar(X_bo, Y_bo, bounds, seed=None):
    if seed is not None:
        torch.manual_seed(seed)

    rango = bounds[1] - bounds[0]
    X_bo_norm = (X_bo - bounds[0]) / rango

    gp = SingleTaskGP(X_bo_norm, Y_bo)
    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)

    log_ei = LogExpectedImprovement(gp, best_f=Y_bo.max())
    unit_bounds = torch.stack([torch.zeros_like(rango), torch.ones_like(rango)]).double()
    candidato_norm, _ = optimize_acqf(
        log_ei, bounds=unit_bounds, q=1, num_restarts=10, raw_samples=100
    )

    candidato_real = candidato_norm * rango + bounds[0]
    return candidato_real

In [ ]:
# ------------------------------------------------------------
# Cell 43: close the 'Hone' leak -- re-clean the
# Wikipedia text already committed to the corpus, re-save, re-chunk,
# re-embed. Same pattern as Cell 19 (strip_citation_lines),
# generalized to any leakage_indicators term.
# ------------------------------------------------------------
import re

def strip_leakage_lines(text, indicadores):
    """Elimina cualquier línea que contenga literalmente uno de los
    términos de fuga conocidos (nombres de autor, parámetros del
    simulador, jerga específica del paper fuente). Igual de agresivo
    que strip_citation_lines, pero generalizado a la lista completa
    de leakage_indicators en vez de solo el patrón de cita."""
    patron = re.compile("|".join(re.escape(v) for v in indicadores.values()), re.IGNORECASE)
    lineas_eliminadas = []
    lineas_mantenidas = []
    for linea in text.split("\n"):
        if patron.search(linea):
            lineas_eliminadas.append(linea)
        else:
            lineas_mantenidas.append(linea)
    return "\n".join(lineas_mantenidas).strip(), lineas_eliminadas

wikipedia_url_en_corpus = [u for u in snar_general_mechanism_corpus if "wikipedia" in u.lower()]
assert len(wikipedia_url_en_corpus) == 1, "Esperaba exactamente la URL de Wikipedia en el corpus"
wikipedia_url_en_corpus = wikipedia_url_en_corpus[0]

texto_antes = snar_general_mechanism_corpus[wikipedia_url_en_corpus]
texto_limpio, eliminadas = strip_leakage_lines(texto_antes, leakage_indicators)

print(f"Líneas eliminadas por fuga ({len(eliminadas)}):")
for l in eliminadas:
    print(f"  > {l[:150]}")

snar_general_mechanism_corpus[wikipedia_url_en_corpus] = texto_limpio

# Final check: repeat the leakage check over the COMPLETE corpus
# (not just the Wikipedia page) before considering the corpus finalized.
full_corpus_text = "\n".join(snar_general_mechanism_corpus.values())
hits_finales = {label: needle.lower() in full_corpus_text.lower()
                for label, needle in leakage_indicators.items()}
if any(hits_finales.values()):
    print("\n⚠️  Sigue habiendo fuga tras la limpieza:", [k for k, v in hits_finales.items() if v])
else:
    print("\n✅ Corpus completo verificado sin fuga -- ahora sí, seguro para el Critic.")

with open("snar_grounding_corpus.json", "w", encoding="utf-8") as f:
    json.dump(snar_general_mechanism_corpus, f, ensure_ascii=False, indent=2)

# Re-chunk + re-embed (the corpus changed size)
snar_corpus_chunks = []
for url, text in snar_general_mechanism_corpus.items():
    for chunk in chunk_text(text):
        snar_corpus_chunks.append({"source": url, "text": chunk})

chunk_texts = [c["text"] for c in snar_corpus_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, normalize_embeddings=True)
print(f"\nCorpus final re-indexado: {len(snar_corpus_chunks)} chunks, embeddings {chunk_embeddings.shape}")


In [ ]:
# ============================================================
# Actual fix: Constant Liar in correr_campania_snar's retries.
# Replaces the dia14_extension_n5.py version (which already had the
# RNG fix in the main proposal, but still lacked Constant Liar
# in the retries -- this file keeps both fixes together).
#
# Same pattern as deliberar_multiagente in dia11_prompt_sensitivity.ipynb:
# local temporary copies (X_bo_temp/Y_bo_temp) that ONLY exist
# during this iteration's retry loop -- they never leak
# into the real X_bo/Y_bo, which are only updated at the end with the
# already-approved decision (or the fallback).
# ============================================================

def correr_campania_snar(metodo, seed_experimento, n_iteraciones, max_rechazos=3):
    """metodo: 'bo_puro' | 'multiagent_norag' | 'multiagent_grounded'"""
    torch.manual_seed(seed_experimento)
    X_init = torch.rand(N_INIT_SNAR, N_DIMS_SNAR) * (BOUNDS_SNAR[1] - BOUNDS_SNAR[0]) + BOUNDS_SNAR[0]

    rng = np.random.default_rng(seed_experimento)
    historial = []
    Y_init = []
    for i in range(N_INIT_SNAR):
        d = tensor_a_dict_snar(X_init[i])
        sty = objetivo_snar(d, noise_level_pct=NOISE_LEVEL_PCT_SNAR, rng=rng)
        Y_init.append(sty)
        historial.append({**d, "sty": sty, "fuente": "init", "rechazos": 0,
                           "concern_level": None, "grounding_usado": []})

    X_bo = X_init.clone().double()
    Y_bo = torch.tensor(Y_init, dtype=torch.float64).unsqueeze(-1)

    registros = []

    for iteracion in range(n_iteraciones):
        candidato_tensor = proponer_candidato_bo_snar(
            X_bo, Y_bo, BOUNDS_SNAR, seed=seed_experimento * 1000 + iteracion
        )
        candidato_dict = tensor_a_dict_snar(candidato_tensor)

        if metodo == 'bo_puro':
            decision = candidato_dict
            fuente = 'bo_puro'
            rechazos = 0
            concern_level = None
            grounding_usado = []
        else:
            # -- Constant Liar: temporary copies, only for this iteration's
            # retry loop. Discarded when the loop exits;
            # never touch the real X_bo/Y_bo. --
            X_bo_temp = X_bo.clone()
            Y_bo_temp = Y_bo.clone()

            intentos = 0
            aprobado = False
            propuesta = None
            ultimo_veredicto = None

            while not aprobado and intentos < max_rechazos:
                propuesta = llamar_proposer_snar(candidato_dict, historial,
                                                  seed_experimento, iteracion * 10 + intentos)
                if metodo == 'multiagent_grounded':
                    veredicto = llamar_critic_snar_v2(propuesta, historial, BOUNDS_SNAR,
                                                       seed_experimento, iteracion * 10 + intentos)
                elif metodo == 'multiagent_norag':
                    veredicto = llamar_critic_snar_no_rag(propuesta, historial, BOUNDS_SNAR,
                                                           seed_experimento, iteracion * 10 + intentos)
                else:
                    raise ValueError(f"Método desconocido: {metodo}")

                ultimo_veredicto = veredicto
                if veredicto['approved']:
                    aprobado = True
                else:
                    # -- Constant Liar: fantasizes the rejected point (the
                    # Proposer's proposal, not the raw BO candidate --
                    # same criterion as dia11) as already evaluated, with the
                    # worst outcome seen so far. This DOES change the
                    # acquisition surface for the retry. --
                    x_rechazado = dict_a_tensor_snar(propuesta)
                    y_fantasma = Y_bo_temp.min().reshape(1, 1)
                    X_bo_temp = torch.cat([X_bo_temp, x_rechazado])
                    Y_bo_temp = torch.cat([Y_bo_temp, y_fantasma])

                    intentos += 1
                    candidato_tensor = proponer_candidato_bo_snar(
                        X_bo_temp, Y_bo_temp, BOUNDS_SNAR,
                        seed=seed_experimento * 100 + iteracion * 10 + intentos
                    )
                    candidato_dict = tensor_a_dict_snar(candidato_tensor)

            if aprobado:
                decision_cruda = llamar_verifier_snar(propuesta, historial, seed_experimento, iteracion)
                decision, recortes =clip_a_bounds_snar(decision_cruda, BOUNDS_SNAR)
                if recortes:
                    print(f" Verifier fuera de bounds, recortado:{recortes}")
                fuente = metodo
            else:
                decision = candidato_dict
                decision['final_reasoning'] = 'Fallback: agentes no llegaron a consenso'
                fuente = 'bo_fallback'

            rechazos = intentos
            concern_level = ultimo_veredicto.get('concern_level') if ultimo_veredicto else None
            grounding_usado = ultimo_veredicto.get('grounding_usado', []) if ultimo_veredicto else []

        x_final = dict_a_tensor_snar(decision)
        sty_final = objetivo_snar(decision, noise_level_pct=NOISE_LEVEL_PCT_SNAR, rng=rng)

        # -- The real X_bo/Y_bo are only updated here, with the
        # final decision (approved or fallback) -- never with the phantom points. --
        X_bo = torch.cat([X_bo, x_final])
        Y_bo = torch.cat([Y_bo, torch.tensor([[sty_final]], dtype=torch.float64)])

        fila = {
            "seed": seed_experimento, "metodo": metodo, "iteracion": iteracion,
            "tau": decision["tau"], "equiv_pldn": decision["equiv_pldn"],
            "conc_dfnb": decision["conc_dfnb"], "temperature": decision["temperature"],
            "sty": sty_final, "fuente": fuente, "rechazos": rechazos,
            "concern_level": concern_level,
            "grounding_usado": ";".join(grounding_usado) if grounding_usado else None,
        }
        registros.append(fila)
        historial.append({**decision, "sty": sty_final, "fuente": fuente,
                           "rechazos": rechazos, "concern_level": concern_level,
                           "grounding_usado": grounding_usado})

    return registros

print("correr_campania_snar redefinida con Constant Liar real en los reintentos.")
print("No relances el piloto todavía -- confírmame que esto es lo que esperabas")
print("antes de que te pase el lanzador n=5 x 12 iteraciones actualizado.")

In [ ]:
import pandas as pd
import pickle
import time

SEEDS_PILOTO_SNAR = [0, 1, 2, 3, 4]
N_ITERACIONES_PILOTO_SNAR = 12
METODOS_PILOTO_SNAR = ['bo_puro', 'multiagent_norag', 'multiagent_grounded']
CHECKPOINT_PATH_SNAR = "checkpoint_piloto_snar.pkl"

todos_los_registros = []
t0 = time.time()
for metodo in METODOS_PILOTO_SNAR:
    for seed in SEEDS_PILOTO_SNAR:
        print(f"\n>>> Corriendo {metodo}, seed={seed}...")
        registros = correr_campania_snar(metodo, seed, N_ITERACIONES_PILOTO_SNAR)
        todos_los_registros.extend(registros)
        print(f"    mejor STY = {max(r['sty'] for r in registros):.2f}  "
              f"(tiempo acumulado: {time.time()-t0:.0f}s)")

        # Checkpoint after every complete (method, seed) run -- if the job dies
        # midway (e.g. HPC walltime limit), we keep everything that has
        # already finished, same as in dia10.
        with open(CHECKPOINT_PATH_SNAR, "wb") as f:
            pickle.dump(todos_los_registros, f)

df_piloto_snar = pd.DataFrame(todos_los_registros)
df_piloto_snar.to_csv("piloto_snar_multiseed_n5.csv", index=False)
print(f"\nGuardado en piloto_snar_multiseed_n5.csv -- {len(df_piloto_snar)} filas, "
      f"tiempo total: {time.time()-t0:.0f}s")

# ------------------------------------------------------------
# Cell 47: multi-seed pilot summary -- best-so-far per seed,
# rejection/fallback rate, and most importantly to close the
# question from dia11: distribution of concern_level when it DOES approve.
# ------------------------------------------------------------
def running_best_df(sty_series):
    return sty_series.cummax()

df_piloto_snar['best_so_far'] = (
    df_piloto_snar.sort_values(['metodo', 'seed', 'iteracion'])
    .groupby(['metodo', 'seed'])['sty']
    .transform(running_best_df)
)

resumen_final = (
    df_piloto_snar[df_piloto_snar['iteracion'] == N_ITERACIONES_PILOTO_SNAR - 1]
    .groupby('metodo')['best_so_far']
    .agg(['mean', 'std', 'count'])
)
print("Best-so-far final por método (media sobre seeds):")
print(resumen_final)

print("\nTasa de rechazo por método (fracción de intentos rechazados por el Critic):")
print(df_piloto_snar[df_piloto_snar['metodo'] != 'bo_puro'].groupby('metodo')['rechazos'].apply(lambda s: (s > 0).mean()))

print("\nDistribución de concern_level CUANDO aprueba (solo multiagent_grounded vs multiagent_norag):")
print(df_piloto_snar[df_piloto_snar['metodo'] != 'bo_puro'].groupby(['metodo', 'concern_level']).size())

# Chart: best-so-far curves, mean +/- individual seeds, per method
plt.figure(figsize=(9, 5.5))
colores = {'bo_puro': 'C0', 'multiagent_norag': 'C1', 'multiagent_grounded': 'C2'}
for metodo in METODOS_PILOTO_SNAR:
    sub = df_piloto_snar[df_piloto_snar['metodo'] == metodo]
    media = sub.groupby('iteracion')['best_so_far'].mean()
    plt.plot(media.index, media.values, marker='o', linewidth=2.5,
              label=f"{metodo} (media, n={len(SEEDS_PILOTO_SNAR)})", color=colores[metodo])
    for seed in SEEDS_PILOTO_SNAR:
        s = sub[sub['seed'] == seed]
        plt.plot(s['iteracion'], s['best_so_far'], alpha=0.25, linewidth=1, color=colores[metodo])
plt.xlabel('Número de evaluaciones post-init')
plt.ylabel('Mejor STY encontrado')
plt.title(f'SNAr: piloto multi-seed (n={len(SEEDS_PILOTO_SNAR)}) -- media y seeds individuales')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('piloto_snar_multiseed_comparacion.png', dpi=150)
plt.show()


In [ ]:
import pandas as pd
df_rescate = pd.DataFrame(todos_los_registros)
df_rescate.to_csv("piloto_snar_n5_parcial_rescatado.csv", index=False)
print(df_rescate.groupby('metodo').size())